In [1]:
import jax.numpy as jnp
from scipy.stats import norm, multivariate_normal, binom
from IPython.display import Markdown, display
from jax import random

In [2]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

# Bayesian Multiclass Classification

In [ ]:
######## Functions #########
def log_npdf(x, m, v):
    return -0.5*(x-m)**2/v - 0.5*jnp.log(2*jnp.pi*v)

# convert from class label to one-hot encoding
def to_onehot(y, num_classes):
    return jnp.column_stack([1.0*(y==value) for value in jnp.arange(num_classes)])


# softmax transformation
def softmax(a_, axis=1):
    max_val = jnp.max(a_, axis=axis)                # get maximum value along axis
    a = a_ - jnp.expand_dims(max_val, axis=axis)    # subtract max value for numerical stability
    exp_a = jnp.exp(a)                                
    return exp_a/jnp.sum(exp_a, axis=axis)[:, None]


In [9]:
# linear model with intercept and slope
design_matrix = lambda x: jnp.column_stack((jnp.ones(len(x)), x))

#Define training data
Xtrain  = jnp.array([[5.89],[11]])
Phi_train = design_matrix(Xtrain)

#Define training targets
ytrain = jnp.array([0,1])
num_classes = len(jnp.unique(ytrain))
ytrain_onehot = to_onehot(ytrain, num_classes)

# Define test data
Xtest = jnp.array([[0,1]])
Phi_test = design_matrix(Xtest)

######## Parameters ########
alpha = 1/4
D = 2
w_MAP = jnp.array([[1,2], [2, 8]])

## Log Likelihood - $\log{p(\bf{y} | \bf{w})}$

### Equation

Likelihood:

$$
\begin{align*}
p(\mathbf{y}|\mathbf{W}) =\prod_{n=1}^N p(y_n|\mathbf{W}, \mathbf{x}_n) = \prod_{n=1}^N \text{Cat}(y_n|\text{Softmax}(\mathbf{W}\phi(\mathbf{x}_n)))
\end{align*}
$$

Log Likelihood:
$$
\begin{align*}
\log{p(\mathbf{y}|\mathbf{W})} = \sum_{n=1}^N \text{Cat}(y_n|\text{Softmax}(\mathbf{W}\phi(\mathbf{x}_n)))
\end{align*}
$$

### Code

In [73]:
def log_likelihood(Xtrain, ytrain_onehot, w_flat):
    """ Evaluates the log likelihood for dataset (self.X, self.y) using a Categorical distribution with softmax inverse link function
        The function accepts the argument w_flat, which is a flattened version of W, such that the shape of w_flat is (T,), where T = num_classes x D is the total number of parameters.
        The return value of the function must be a scalar.
    """
    
    # reshape from flat vector to matrix of size num_classes by D
    W = w_flat.reshape((num_classes, D))
    
    # compute values for each latent function
    y_all = Xtrain @ W.T

    # normalize using softmax
    p_all = softmax(y_all)
    
    # evaluate 
    loglik_val =  jnp.sum(ytrain_onehot*jnp.log(p_all))
    
    return loglik_val

log_l = log_likelihood(Phi_train, ytrain_onehot, w_MAP)
l = jnp.exp(log_l)
display(Markdown(rf"The log-likelihood is $log(p(y | w)) = {log_l:.2f}$"))
display(Markdown(rf"The likelihood is $p(y | w) = {l:.2f}$"))

The log-likelihood is $log(p(y | w)) = -36.34$

The likelihood is $p(y | w) = 0.00$

## Log Joint Probability - $\log{p(\bf{y},\bf{w})}$

### Equation

Joint Probability

$$
\begin{align*}
p(\mathbf{y}, \mathbf{W}) &= p(\mathbf{y}|\mathbf{W})p(\mathbf{W})=\prod_{n=1}^N \text{Cat}(y_n|\text{Softmax}(\mathbf{W}\phi(\mathbf{x}_n)))\prod_{i=1}^K \mathcal{N}(\mathbf{w}_i|\mathbf{0}, \alpha^{-1}\mathbf{I}).
\end{align*}
$$

Log Joint Probability

$$
\begin{align*}
\log{p(\mathbf{y}, \mathbf{W})} &= \log{p(\mathbf{y}|\mathbf{W})} + \log{p(\mathbf{W})}=\sum_{n=1}^N \text{Cat}(y_n|\text{Softmax}(\mathbf{W}\phi(\mathbf{x}_n))) + \sum_{i=1}^K \mathcal{N}(\mathbf{w}_i|\mathbf{0}, \alpha^{-1}\mathbf{I}).
\end{align*}
$$

### Code

In [28]:
def log_joint(X, ytrain_onehot, w):
    log_prior = jnp.sum(log_npdf(w, 0, 1./alpha))
    return log_prior + log_likelihood(X, ytrain_onehot, w)

log_j = log_joint(Phi_train, ytrain_onehot, w_MAP)
display(Markdown(rf"The log joint probability is $p(y, w) = {log_j:.4f}$"))

The log joint probability is $p(y, w) = -51.9133$

## Posterior Distribution

Be aware the code below assumes we're given the mean and covariance matrix of the posterior approximation, which else would have to be found using a minimization algorithm.

### Code

In [40]:
def predict_f(m, S, X_star):
    """ computes the posterior distribution of f_i(x, w) = w_i^T phi(x^*) for all K classes

        Arguments:
        X_star            --         PxD prediction points

        Returns
        mu_f_all_classes  --         posterior mean of f for all classes (shape: P x K)
        var_f_all_classes  --        posterior variance of f for all classes (shape: P x K)
        """
    
    # get relevant part for each of the K linear models
    Si = [S[i*D:(i+1)*D, i*D:(i+1)*D] for i in range(num_classes)]

    # compute mean and variance for each function
    mu_f_all_classes = X_star @ m.T
    var_f_all_classes = jnp.squeeze(jnp.stack([jnp.diag(X_star@Si[i]@X_star.T) for i in range(num_classes)], axis=1))

    return mu_f_all_classes, var_f_all_classes

m = jnp.array([[2,2], [2,2]])
S = jnp.array([[2,1,2,3], [1,2,3,4], [2,3,6,4], [1,4,6,5]])

mu_f_all_classes, var_f_all_classes = predict_f(m, S, Xtest)

display(Markdown(rf"The posterior distribution is $p(f | x_*, y) = \mathcal{{N}}(f \mid \mu, \sigma^2)$ where $\mu = {to_latex_vector(mu_f_all_classes.flatten())}$ and $\sigma^2 = {to_latex_vector(var_f_all_classes.flatten())}$"))

The posterior distribution is $p(f | x_*, y) = \mathcal{N}(f \mid \mu, \sigma^2)$ where $\mu = \begin{bmatrix}2.00 \\ 2.00\end{bmatrix}$ and $\sigma^2 = \begin{bmatrix}2.00 \\ 5.00\end{bmatrix}$

## Posterior predictive - $p(y^{*} | \bf{w}, \bf{x^{*}})$

### Equation

Posterior Predictive Distribution

$$p(y^*|\mathbf{y}, \mathbf{x}^*) = \text{Cat}(y^*|\boldsymbol{\pi}),$$

where the $k$'th entry in  $\boldsymbol{\pi} \in \left[0, 1\right]^K$ is the posterior probability of $y^* = k$, i.e. $\boldsymbol{\pi}_k = p(y^*=k|\mathbf{y}, \mathbf{x}^*)$. 

Posterior Predictive Distribution given the Laplace approximation $q(\mathbf{w})$, approximated using **Monte Carlo sampling**
$$\begin{align*}
\boldsymbol{\pi}_k = p(y^* = k|\mathbf{y}, \mathbf{x}^*) {\approx} \frac{1}{S} \sum_{j=1}^S \text{Softmax}(\mathbf{f}_*^{(i)})_k \tag{P}
\end{align*}$$

### Code

In [115]:
def posterior_predictive(X_star, f_samples):
    
    # compute softmax for all individual samples (shape: P x num_classes x num_samples)
    p_all_samples = jnp.array([softmax(X_star @ f_sample.T, axis=1) for f_sample in f_samples])
    
    # compute mean over Monte Carlo samples  (shape: P x num_classes)
    p_all = p_all_samples.mean(0)
    
    return p_all


f_samples = jnp.array([jnp.array([[-0.15, -1.92], [3.2, 0.45], [1.37, 0.8]]), jnp.array([[-0.31, -2.03], [2.98, 0.08], [1.03, 1.29]]), jnp.array([[-0.35, -1.98], [3.09, 0.07], [1.3, 0.96]])])
Xtest = design_matrix(jnp.array([-1]))

p_all = posterior_predictive(Xtest, f_samples)
for i, p in enumerate(p_all.flatten()):
    display(Markdown(rf"The predictive distribution is $p(y^{{**}})={i+1} | x^*, y) = {p:.2f}$"))

The predictive distribution is $p(y^{**})=1 | x^*, y) = 0.22$

The predictive distribution is $p(y^{**})=2 | x^*, y) = 0.72$

The predictive distribution is $p(y^{**})=3 | x^*, y) = 0.05$

## Entropy (Predictive uncertainty)

### Equation

For a **discrete random variable** entropy is given by:

$$\begin{align*}
\mathcal{H}\left[p(y^*|\mathbf{y}, \mathbf{x}^*)\right] = \mathcal{H}\left[\text{Cat}(y^*|\boldsymbol{\pi})\right]= -\sum_{i=1}^K \boldsymbol{\pi}_k \log \boldsymbol{\pi}_k,
\end{align*}$$
where we use the convention $0\log 0 = 0$ for the entropy calculation. 

### Code

In [ ]:
def compute_entropy(pi):
    """ assumes pi is [N, K] where N is the number of prediction points and K is the number of classes """ 
    log_pi = jnp.where(pi > 0, jnp.log(pi), 0)  # equal to log(p) when p > 0 else 0
    H = -jnp.sum(pi*log_pi, 1)
    return H

entropy = compute_entropy(p_all)
display(Markdown(rf"The entropy is $\mathcal{{H}}\left[p(y^*|\mathbf{{y}}, \mathbf{{x}}^*)\right] = {to_latex_vector(entropy)}$"))

The entropy is $\mathcal{H}\left[p(y^*|\mathbf{y}, \mathbf{x}^*)\right] = \begin{bmatrix}0.72\end{bmatrix}$

## Confidence

### Equation

$$
\mathcal{C}\left[p(y^*=k|\mathbf{y}, \mathbf{x}^*)\right] = \max\limits_{k\in\{ 1, \dots, K \}} p(y^*=k|\mathbf{y}, \mathbf{x}^*).
$$

### Code

In [118]:
def compute_confidence(pi):
    """ assumes pi is [N, K] where N is the number of prediction points and K is the number of classes """
    return jnp.max(pi, 1)

confidence = compute_entropy(p_all)
display(Markdown(rf"The entropy is $\mathcal{{C}}\left[p(y^*=k|\mathbf{{y}}, \mathbf{{x}}^*)\right] = {to_latex_vector(confidence)}$"))

The entropy is $\mathcal{C}\left[p(y^*=k|\mathbf{y}, \mathbf{x}^*)\right] = \begin{bmatrix}0.72\end{bmatrix}$

## Total uncertainty, aleatoric uncertainty, and epistemic uncertainty

### Equation

$$\begin{align*}
\boldsymbol{\pi} &= \frac{1}{S}\sum_{i=1}^S \boldsymbol{\pi}^{(i)},\\
\mathcal{H}_{\text{total}} &=\mathcal{H}\left[\text{Cat}(y^*|\boldsymbol{\pi})\right]\tag{TU},\\
\mathcal{H}_{\text{aleatoric}} &=  \frac{1}{S}\sum_{i=1}^S \mathcal{H}\left[\text{Cat}(y^*|\boldsymbol{\pi}^{(i)})\right]\tag{AU},\\
\mathcal{H}_{\text{epistemic}} &= \mathcal{H}_{\text{total}} - \mathcal{H}_{\text{aleatoric}}. \tag{EU}
\end{align*}$$


### Code

In [108]:
def compute_uncertainties(pi_samples):
    # pi_samples = softmax probabilities per sample (shape: len of Xstar x num_classes x num_samples)
    H_total = compute_entropy(jnp.mean(pi_samples, axis=2))      
    H_aleatoric = jnp.mean(compute_entropy(pi_samples), axis=1)  
    H_epistemic = H_total - H_aleatoric   
    return H_total, H_aleatoric, H_epistemic

pi_samples = jnp.array([softmax(Xtest @ f_sample.T, axis=1) for f_sample in f_samples])
H_total, H_aleatoric, H_epistemic = compute_uncertainties(pi_samples)

## Expected Utility

### Equation

\begin{align*}
\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=k)\right] 
= \sum_{c=1}^{K} \mathcal{U}(y^*=c,\, \hat{y}=k)\, p(y^*=c \mid \mathbf{y}, \mathbf{x}^*)
\end{align*}

### Code

In [125]:
def compute_expected_utility(U, phat):
    """ computes the expected utility for a multi-class classification problem with K classes for utility matrix U and posterior predictive probabilities phat 
        
        Arguments
        U               --      Utility matrix (shape: [K x K])
        phat            --      Posterior predictive probabilities (shape: [P x K]), where P is the number of prediction points

        expected_util   --      Expected utility for each class for each point in phat (shape: P x K)           
           """
    

    expected_util = phat@U  

    return expected_util

U = jnp.array([[.5, .5, .9], [.6, .7, .8], [.5, .5, .9]])

expected_util = compute_expected_utility(U, p_all)

display(Markdown(rf"The expected utility is"))
for i, p in enumerate(expected_util.flatten()):
    display(Markdown(rf"$\mathbb{{E}}_{{p(y^*|\mathbf{{y}}, \mathbf{{x}}^*)}}\left[\mathcal{{U}}(y^*, \hat{{y}}={i+1})\right]  = {p:.2f}$"))

The expected utility is

$\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=1)\right]  = 0.57$

$\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=2)\right]  = 0.64$

$\mathbb{E}_{p(y^*|\mathbf{y}, \mathbf{x}^*)}\left[\mathcal{U}(y^*, \hat{y}=3)\right]  = 0.83$